# Chapitre 1 — Lire les données · Corrigé

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch01_corrige.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Chaque exercice reprend l'énoncé du livre. La réponse commence par le raisonnement sur papier, puis montre son équivalent en Python.

**Convention.** Comme dans le livre, la variance, l'écart-type, le score z et la covariance divisent par $n$ (`ddof=0`), sauf mention explicite.

**Sommaire.** Données · Les calculs des leçons · Exercices 1.1 à 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

Les identifiants (D0.1, D1.1…) sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 1, recopiées des fichiers CSV du livre (ne pas modifier)

# D0.1 — les six logements de l'ouverture
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — prix de vente des T2 (k€), leçon 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — registre des ventes du quartier Sud, leçons 1.2 à 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — prix au m² selon l'étage (k€/m²), leçon 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — délais de vente de trois agences (jours), exercice 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — loyers mensuels des studios (€), exercice 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Outils fournis : vous n'avez pas à les modifier

def quartiles_livre(valeurs):
    """Q1 et Q3 selon la règle du livre (équation 1.5) : rangs ⌈n/4⌉ et ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # arrondi à l'entier supérieur de n/4
    rang_q3 = -(-3 * n // 4)      # arrondi à l'entier supérieur de 3n/4
    return x[rang_q1 - 1], x[rang_q3 - 1]   # rang 1 = position 0 en Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Une rangée de points par série, comme les figures de la leçon 1.1.
    bandes : dictionnaire {nom: (moyenne, écart-type)} pour dessiner moyenne ± écart-type."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 1.1 à 1.5. Exécutez-les et comparez avec le livre. Ici, `np.std` et `np.var` sont appelés avec `ddof=0`, c'est-à-dire avec une division par $n$, comme dans le chapitre.

In [ ]:
# Leçon 1.1 — moyenne, variance et écart-type (division par n)
for nom, serie in [('Nord', nord), ('Sud', sud)]:
    print(f'{nom} : moyenne = {np.mean(serie):g} k€, variance = {np.var(serie, ddof=0):g} k€², '
          f'écart-type = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Leçon 1.2 — effectifs par classe DPE (variable qualitative ordinale)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Leçon 1.3 — moyenne, médiane et quartiles des délais
d = registre['delai_jours']
print('moyenne :', d.mean(), ' médiane :', d.median())
print('sans S13 : moyenne', d[registre['vente'] != 'S13'].mean(),
      ' médiane', d[registre['vente'] != 'S13'].median())
print('quartiles, règle du livre (équation 1.5) :', quartiles_livre(d))
print('quartiles, NumPy par défaut (interpolation) :', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Leçon 1.4 — scores standardisés (équation 1.6, écart-type avec division par n)
for nom, serie, vente in [('Nord', nord, 188), ('Sud', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom} : z de la série = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z de la vente à {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Leçon 1.5 — covariance et corrélation des cinq T2 du Sud (division par n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('covariance :', cov, 'm²·k€   r :', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r par np.corrcoef :', np.corrcoef(x, y)[0, 1].round(4))
print('r étage / prix au m² :', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Exercice 1.1 — Lire un graphique

*Données D1.E1. Notions : moyenne et écart-type (leçon 1.1).* Le graphique ci-dessous reproduit celui de l'exercice 1.1 du livre.

In [ ]:
points_par_rangee({f'Agence {a}': v for a, v in delais_agences.items()},
                  'Délai de vente (jours)')
plt.xlim(0, 60)
plt.show()

**a) et b)** Estimez à l'œil le délai moyen de chaque agence, puis rangez les agences de la moins dispersée à la plus dispersée.

In [ ]:
moyenne_estimee = {'P': 30, 'Q': 30, 'R': 30}   # en jours
ordre_dispersion = ['P', 'Q', 'R']

In [ ]:
# Vérification (division par n)
for a, v in delais_agences.items():
    print(f'Agence {a} : moyenne = {np.mean(v):g} j, écart-type = {np.std(v, ddof=0):.2f} j')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Complétez la cellule précédente.')
else:
    print('Ordre de dispersion :', 'correct' if ordre_dispersion == ordre_reel else 'à revoir')

**c)** Comptez, pour Q et pour R, les ventes conclues à 2 jours ou moins de la moyenne.

In [ ]:
proches_Q = 1
proches_R = 3

In [ ]:
# Vérification
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agence {a} : pas encore complété.')
    else:
        print(f"Agence {a} : {'correct' if la_votre == vrai else 'à revoir'}")

### Corrigé de l'exercice 1.1

**a)** Les trois séries sont symétriques autour de 30 jours. Le délai moyen vaut **30 jours** pour chaque agence, ce que confirme le calcul : (28 + 29 + 30 + 31 + 32) / 5 = 150 / 5 = 30, et de même pour Q et R.

**b)** De la moins à la plus dispersée : **P, Q, R**. Les points de P tiennent entre 28 et 32 jours. Ceux de Q s'étalent régulièrement de 20 à 40 jours. Ceux de R vont de 10 à 50 jours. Écarts-types (division par $n$) : P ≈ 1,41 j ; Q ≈ 7,07 j ; R ≈ 12,71 j.

**c)** Q : **1** vente (30 jours). R : **3** ventes (28, 30 et 32 jours). R a pourtant le plus grand écart-type. Ses deux ventes extrêmes, à 10 et 50 jours, sont à 20 jours de la moyenne. Leurs carrés valent 400 chacun : la somme des carrés de R vaut 400 + 4 + 0 + 4 + 400 = 808, contre 100 + 25 + 0 + 25 + 100 = 250 pour Q. L'écart-type résume tous les écarts, et les carrés donnent un poids important aux plus grands. Trois ventes proches du centre ne compensent pas deux ventes très éloignées.

*Éléments attendus :* le classement P, Q, R ; les deux comptages ; le rôle des deux valeurs extrêmes et des carrés dans le calcul.

## Exercice 1.2 — Deux résidences, un même loyer moyen

*Données D1.E2. Notions : moyenne, variance et écart-type (leçon 1.1).* Camille ne peut pas dépasser 570 € par mois. Les questions a) à d) se font d'abord sur papier, avec le tableau de l'exercice 1.2 du livre. Ce notebook sert à vérifier vos réponses, puis à programmer l'écart-type (question e) et à comparer deux conventions (question f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Loyer mensuel d'un studio (€)")
plt.axvline(570, color='black', linestyle=':', label='budget de Camille : 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Lire les loyers

Combien de studios Camille peut-elle louer dans chaque résidence avec 570 €, puis avec 620 € ?

In [ ]:
accessibles = {
    570: {'Les Tilleuls': 0, 'Le Port': 2},
    620: {'Les Tilleuls': 6, 'Le Port': 4},
}

In [ ]:
# Vérification : `<=` produit des vrai/faux, np.sum compte les « vrai »
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'pas encore complété' if la_votre is None else ('correct' if la_votre == vrai else 'à revoir')
        print(f'Budget {budget} €, {nom} : {etat}')

### Corrigé de 1.2 a)

**En lisant la liste.** Avec 570 €, Camille ne trouve **aucun** studio aux Tilleuls : le moins cher coûte 585 €. Au Port, **deux** studios sur six lui sont accessibles, à 520 et 560 €. La résidence où elle a une chance est donc celle dont les loyers sont les plus dispersés. Avec 620 €, la situation s'inverse : les **six** studios des Tilleuls lui sont accessibles, contre **quatre** sur six au Port (520, 560, 590 et 620 €). Les deux moyennes égales ne laissaient rien deviner de ces différences.

### b) Les deux moyennes

**Sur papier.** Les Tilleuls : 585 + 590 + 600 + 605 + 605 + 615 = 3 600, et 3 600 / 6 = **600 €**. Le Port : 520 + 560 + 590 + 620 + 640 + 670 = 3 600, et 3 600 / 6 = **600 €**.

**En Python.** `np.mean` fait la même somme, puis la même division.

In [ ]:
moyenne_tilleuls_main = 600
moyenne_port_main = 600

In [ ]:
# Vérification
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom} : complétez la cellule précédente.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom} : {'correct' if ok else 'à revoir'} (votre valeur : {la_votre} €)")

### c) Variance et écart-type à la main (division par $n = 6$)

**Les Tilleuls** ($\bar{x} = 600$ €)

| $x_i$ (€) | 585 | 590 | 600 | 605 | 605 | 615 | Somme |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −15 | −10 | 0 | +5 | +5 | +15 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 225 | 100 | 0 | 25 | 25 | 225 | 600 |

Variance : 600 / 6 = **100 €²**. Écart-type : √100 = **10 €**.

**Le Port** ($\bar{x} = 600$ €)

| $x_i$ (€) | 520 | 560 | 590 | 620 | 640 | 670 | Somme |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −80 | −40 | −10 | +20 | +40 | +70 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 6 400 | 1 600 | 100 | 400 | 1 600 | 4 900 | 15 000 |

Variance : 15 000 / 6 = **2 500 €²**. Écart-type : √2 500 = **50 €**.

*Deux contrôles en cours de route.* Dans chaque tableau, la ligne des écarts doit totaliser 0 (leçon 1.1). Ensuite, l'écart-type doit rester du même ordre que les écarts du tableau : 10 € pour des écarts de 0 à 15 €, 50 € pour des écarts de 10 à 80 €.

**En Python.** `np.var(..., ddof=0)` et `np.std(..., ddof=0)` divisent par $n$.

In [ ]:
variance_tilleuls_main = 100   # en €²
variance_port_main = 2500      # en €²
ecart_type_tilleuls_main = 10  # en €
ecart_type_port_main = 50      # en €

In [ ]:
# Vérification (division par n : ddof=0)
vos_valeurs = {
    'variance Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'variance Port': (variance_port_main, np.var(port, ddof=0)),
    'écart-type Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'écart-type Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom} : pas encore complété.')
    else:
        print(f"{nom} : {'correct' if np.isclose(la_votre, attendu) else 'à revoir'}")

### d) Interpréter

*Exemple de réponse.* Aux Tilleuls, les loyers s'écartent habituellement d'environ 10 € de 600 € : ils sont presque tous identiques. Au Port, l'écart habituel est d'environ 50 € : on y trouve des studios nettement moins chers et nettement plus chers que la moyenne. L'écart-type ne suffit pas pour répondre à la question a). Il décrit l'étalement en général, mais ne dit pas si un studio précis coûte 570 € ou moins. Un écart-type de 50 € autour de 600 € rend plausible un loyer sous 570 €. Seule la liste permet de compter qu'il y en a exactement deux, à 520 et 560 €.

*Éléments attendus :* une phrase par résidence, avec l'unité ; la distinction entre une mesure de dispersion et la question « un studio entre-t-il dans le budget ? » ; le recours à la liste des loyers.

### e) Programmer l'écart-type

Complétez la fonction en suivant l'équation 1.3 du livre, avec une division par $n$ :

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

N'utilisez ni `np.std` ni `np.var` : le but est d'écrire chaque étape vous-même.

In [ ]:
# Exercice 1.2 e)
def ecart_type(valeurs):
    """Écart-type d'une série, avec division par n (équation 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = np.sum(x) / n               # équation 1.1
    ecarts = x - moyenne                  # une soustraction par valeur
    variance = np.sum(ecarts ** 2) / n    # équation 1.2
    return np.sqrt(variance)              # retour aux euros

Chaque ligne de la fonction reprend une colonne du tableau de la question c) : les écarts, leurs carrés, la moyenne de ces carrés, puis la racine carrée pour revenir en euros. Erreurs fréquentes : prendre la racine avant la moyenne ; calculer `np.sum(ecarts) / n`, qui donne toujours 0 (leçon 1.1).

In [ ]:
# Vérification : exécutez cette cellule sans la modifier
if ecart_type(tilleuls) is None:
    print('La fonction renvoie encore None : complétez-la.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'correct' if np.isclose(obtenu, attendu) else f'à revoir (attendu : {attendu})'
        print(f'{nom} : {obtenu:.2f} € → {etat}')

### f) Approfondissement : diviser par $n$ ou par $n - 1$

Par défaut, la méthode `.std()` de pandas divise la somme des carrés par $n - 1$ (`ddof=1`). NumPy divise par $n$ par défaut (`ddof=0`), la convention du chapitre.

**Sur papier, avec division par $n - 1 = 5$.** Tilleuls : 600 / 5 = 120, puis √120 ≈ **10,95 €**. Port : 15 000 / 5 = 3 000, puis √3 000 ≈ **54,77 €**.

Les deux conventions sont correctes, mais elles ne répondent pas à la même question. Diviser par $n$ décrit les six studios eux-mêmes. Diviser par $n - 1$ sert lorsque ces six studios représentent un ensemble plus large. Le chapitre 6 expliquera ce choix. Avec `.std(ddof=0)`, pandas retrouve 10 et 50 €. Dans un calcul, annoncez toujours la convention utilisée.

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, par défaut (division par n − 1) :', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (division par n)          :', loyers.std(ddof=0).round(2).to_dict())

## Exercice 1.3 — Une vente surprenante ?

*Données D1.1. Notions : score standardisé (leçon 1.4, équation 1.6).* Un T2 vient de se vendre 196 k€. Moyennes et écarts-types : ceux de la leçon 1.1, avec division par $n$.

In [ ]:
z_nord = (196 - 180) / 4     # = 4
z_sud = (196 - 180) / 20     # = 0,8
prix_sud_aussi_exceptionnel = 180 + 4 * 20   # = 260 k€

In [ ]:
# Vérification (écart-type avec division par n)
attendus = {
    'z Nord': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Sud': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'prix Sud équivalent': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'pas encore complété' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'à revoir')
    print(f'{nom} : {etat}')

### Corrigé de l'exercice 1.3

**a)** Au Nord : $z = (196 - 180)/4 = 16/4 =$ **4**. Au Sud : $z = (196 - 180)/20 = 16/20 =$ **0,8**. Les deux scores sont sans unité : ils comptent des écarts-types.

**b)** Elle surprendrait au **Nord**. Sur la figure des scores standardisés de la leçon 1.4, un score de 4 tombe bien au-delà de toutes les ventes du Nord, dont aucune ne dépasse 1,25. Au Sud, 0,8 se situe entre les ventes à 0,75 et 1,25 : une vente ordinaire.

**c)** On cherche le prix du Sud dont le score vaut 4 : $180 + 4 \times 20 =$ **260 k€**. Il faut repartir de l'équation 1.6 et isoler $x$ : $x = \bar{x} + z\,\sigma$.

*Éléments attendus :* les deux scores ; la comparaison en écarts-types plutôt qu'en k€ ; le prix équivalent de 260 k€.

## Exercice 1.4 — Surface et prix vont-ils ensemble ?

*Données D0.1 (les six logements de l'ouverture). Notions : covariance et corrélation (leçon 1.5, équations 1.7 et 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Surface (m²)')
plt.ylabel('Prix de vente (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Les points montent de gauche à droite et longent presque une droite : $r$ sera **positif et très proche de 1**.

**b)** Complétez les lignes ci-dessous. Utilisez une division par $n$ partout.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = (x - x.mean()) * (y - y.mean())          # un produit par logement
covariance = np.sum(produits) / n                   # équation 1.7
r = covariance / (x.std(ddof=0) * y.std(ddof=0))    # équation 1.8

**Sur papier**, la surface moyenne vaut 384 / 6 = 64 m² et le prix moyen 1 334 / 6 ≈ 222,33 k€. La somme des six produits vaut 9 825, d'où une covariance de 9 825 / 6 = **1 637,5 m²·k€**. Les écarts-types valent environ 22,80 m² et 72,15 k€, donc $r ≈ 1637{,}5 / (22{,}80 × 72{,}15) ≈$ **0,996**.

In [ ]:
# Vérification
if r is None or covariance is None:
    print('Complétez la cellule précédente.')
else:
    print('covariance :', round(covariance, 2), 'm²·k€ →',
          'correct' if np.isclose(covariance, 1637.5) else 'à revoir (attendu : 1637,5)')
    print('r :', round(r, 4), '→', 'correct' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'à revoir')

**Piège à connaître.** `np.cov(x, y)` divise par $n - 1$ par défaut et renvoie 1 965, et non 1 637,5. Pour la convention du chapitre, il faut écrire `np.cov(x, y, bias=True)`. Le coefficient $r$, lui, ne dépend pas de la convention, car le même facteur apparaît en haut et en bas (leçon 1.5).

**c)** Il n'y a pas de contradiction. D'abord, $r$ mesure à quel point les points se resserrent autour de *la* droite qui leur convient le mieux (celle que le chapitre 3 apprendra à calculer), pas autour de notre droite tracée à l'œil. Ensuite, la proximité est relative : les prix s'étalent avec un écart-type d'environ 72 k€, et des écarts de 5 à 11 k€ en valeur absolue sont petits à côté de cet étalement. Un $r$ proche de 1 ne garantit donc pas des erreurs petites en k€. Pour connaître la taille des erreurs, il faut les calculer, comme dans l'exercice 1.6.

*Éléments attendus :* « $r$ porte sur la meilleure droite » ; « les écarts sont petits relativement à l'étalement des prix » ; la distinction entre alignement et taille des erreurs en k€.

In [ ]:
print('np.cov par défaut (division par n − 1) :', np.cov(x, y)[0, 1])
print('np.cov avec bias=True (division par n)  :', np.cov(x, y, bias=True)[0, 1])

## Exercice 1.5 — Prédire, puis vérifier

*Données D1.1. Notions : moyenne et écart-type (leçon 1.1), médiane et écart interquartile (leçon 1.3).* Un sixième T2 se vend au Nord pour 240 k€.

**a)** Prédiction attendue : la moyenne augmente nettement, car elle répartit les 240 k€ entre toutes les ventes. La médiane bouge à peine, car la nouvelle vente ne fait que se placer « après le milieu ». L'écart-type augmente beaucoup, car un écart d'environ 50 k€ élevé au carré domine la somme. L'écart interquartile bouge peu, car il ne dépend que des valeurs centrales.

In [ ]:
nord_plus = nord + [240]
print('Nord :', sorted(nord))
print('Nord avec la vente à 240 k€ :', sorted(nord_plus))

**b)** Complétez la fonction `resume`. Elle doit renvoyer les quatre indicateurs, avec l'écart-type divisé par $n$ et les quartiles calculés par la règle du livre (`quartiles_livre`, fournie plus haut).

In [ ]:
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': np.mean(serie),
        'médiane': np.median(serie),
        'écart-type': np.std(serie, ddof=0),   # division par n
        'écart interquartile': q3 - q1,
    }

In [ ]:
tableau = pd.DataFrame({'Nord (5 ventes)': resume(nord), 'Nord + 240 k€ (6 ventes)': resume(nord_plus),
                        'Sud (5 ventes)': resume(sud)})
if tableau.isna().any().any():
    print('Complétez la fonction resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k} : {'correct' if ok else 'à revoir'}")
tableau.round(2)

In [ ]:
# Les deux séries sur le modèle de la figure « moyenne ± écart-type » de la leçon 1.1
if tableau.isna().any().any():
    print('Complétez la fonction resume pour afficher le graphique.')
else:
    points_par_rangee({'Nord': nord, 'Nord + 240': nord_plus},
                      'Prix de vente d\'un T2 (k€)',
                      bandes={'Nord': (np.mean(nord), np.std(nord, ddof=0)),
                              'Nord + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

### Corrigé de l'exercice 1.5

**b) Sur papier.** Avec la vente à 240 k€, la somme passe de 900 à 1 140 k€ pour 6 ventes : moyenne **190 k€** (au lieu de 180). Rangées, les six valeurs sont 174, 177, 181, 183, 185, 240. $n$ est pair, donc la médiane vaut (181 + 183)/2 = **182 k€** (au lieu de 181). Les écarts à 190 sont −16, −13, −9, −7, −5, +50. Leurs carrés valent 256, 169, 81, 49, 25 et 2 500, pour une somme de 3 080. La variance vaut 3 080/6 ≈ 513,3 k€², et l'écart-type **≈ 22,66 k€** (au lieu de 4). Quartiles (équation 1.5) : avant, rangs ⌈5/4⌉ = 2 et ⌈15/4⌉ = 4, soit Q1 = 177 et Q3 = 183, d'où un écart interquartile de **6 k€**. Après, rangs ⌈6/4⌉ = 2 et ⌈18/4⌉ = 5, soit Q1 = 177 et Q3 = 185, d'où **8 k€**.

**c)** Non. Cinq ventes sur six restent à 6 k€ ou moins de 180 k€. Une seule vente exceptionnelle suffit à faire passer l'écart-type de 4 à 22,66 k€, au-dessus des 20 k€ du Sud, car son écart de 50 k€ compte pour 2 500 dans une somme de 3 080. Les mesures robustes racontent autre chose : l'écart interquartile du Nord passe de 6 à 8 k€ et reste très inférieur à celui du Sud (30 k€). Pour un T2 ordinaire du Nord, les prix restent très prévisibles. L'écart-type signale surtout la présence d'une vente hors norme, qu'il faut regarder à part.

*Éléments attendus :* les quatre valeurs avant/après ; le contraste entre indicateurs sensibles (moyenne, écart-type) et robustes (médiane, écart interquartile) ; une conclusion nuancée.

## Exercice 1.6 — Défi : deux droites, une même erreur moyenne

*Données D0.1. Notions : écart signé et valeur absolue (ouverture, leçon 1.1), écart-type (leçon 1.1).* Droite tracée à l'œil : prix prédit = 3 × surface + 30. Droite paresseuse : prix prédit = 222 k€ pour tous les logements.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = prix - (3 * surface + 30)
ecarts_paresseuse = prix - 222

In [ ]:
# Les trois indicateurs de la question b) (écart-type : division par n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Complétez la cellule précédente.')
else:
    indicateurs = pd.DataFrame({
        nom: {'moyenne des écarts signés': e.mean(),
              'moyenne des écarts en valeur absolue': np.abs(e).mean(),
              'écart-type des écarts signés': e.std(ddof=0)}
        for nom, e in [('droite à l\'œil', ecarts_oeil), ('droite paresseuse', ecarts_paresseuse)]
    })
    print('écarts, droite à l\'œil   :', ecarts_oeil)
    print('écarts, droite paresseuse :', ecarts_paresseuse)
    print('vérification :', 'correct' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'à revoir')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="droite à l'œil")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='droite paresseuse')
    ax.set_xlabel('Surface (m²)')
    ax.set_ylabel('Écart observé − prédit (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

### Corrigé de l'exercice 1.6

**a)** Écarts de la droite paresseuse (observé − 222) : −104, −50, −37, +23, +57, +113 k€. Leur somme vaut 2 et leur moyenne ≈ **0,33 k€**. C'est exactement la moyenne des écarts signés de la droite tracée à l'œil (−8, +7, −7, +5, −6, +11 : somme 2, moyenne ≈ 0,33 k€).

**b)**

| | Droite à l'œil | Droite paresseuse |
|:--|--:|--:|
| Moyenne des écarts signés | 0,33 k€ | 0,33 k€ |
| Moyenne des écarts en valeur absolue | 44/6 ≈ 7,33 k€ | 384/6 = 64 k€ |
| Écart-type des écarts signés (division par $n$) | ≈ 7,56 k€ | ≈ 72,15 k€ |

Détail pour la droite à l'œil : la moyenne vaut 1/3. Les écarts à cette moyenne ont des carrés dont la somme vaut 344 − 6 × (1/3)² ≈ 343,33. La variance vaut 343,33/6 ≈ 57,22 et l'écart-type ≈ 7,56 k€.

**c)** La droite tracée à l'œil est bien meilleure : ses erreurs valent environ 7 k€ en valeur absolue, contre 64 k€ pour la droite paresseuse. La moyenne des écarts en valeur absolue et l'écart-type des écarts signés les départagent. La moyenne des écarts signés n'y parvient pas : les deux droites ne penchent d'aucun côté, et les signes se compensent dans les deux cas.

**d)** Les écarts de la droite paresseuse valent $y_i - 222$. Soustraire une même constante décale toute la série sans changer la distance de chaque valeur à la moyenne. La moyenne diminue aussi de 222, donc chaque écart à la moyenne $(y_i - 222) - (\bar{y} - 222) = y_i - \bar{y}$ reste identique. L'écart-type des écarts est donc celui des prix, ≈ 72,15 k€. Une droite qui ignore la surface se trompe en moyenne de l'étalement des prix lui-même. Le chapitre 3 tirera parti de cette comparaison.

*Éléments attendus :* les deux moyennes égales ; deux indicateurs de taille d'erreur nettement différents ; l'explication par la compensation des signes ; l'argument du décalage pour d).

In [ ]:
# Contrôle de d) : l'écart-type des écarts de la droite paresseuse égale celui des prix
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))